# Building an Agent, Step by Step
**LangChain v1 + LangGraph + LangSmith**

Each stage fixes one limitation of the stage before it.

## Stage 0: Setup

In [ ]:
# pip install -U langchain langgraph langchain-openai python-dotenv numpy
from dotenv import load_dotenv
load_dotenv(override=True)   # loads OPENAI_API_KEY and LANGSMITH_* settings from .env

MODEL = "openai:gpt-4o-mini"                  # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## Stage 1: Talk to an LLM
Type `quit` to stop. Tell it your name, then ask it back.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL)

while True:
    user = input("You: ")
    if user == "quit": break
    print("You:", user)
    print("AI:", llm.invoke(user).content)

## Stage 2: Short-term memory
Memory is just *sending the history back every time*. First by hand:

In [ ]:
history = []

while True:
    user = input("You: ")
    if user == "quit": break
    print("You:", user)
    history.append({"role": "user", "content": user})
    reply = llm.invoke(history)
    history.append({"role": "assistant", "content": reply.content})
    print("AI:", reply.content)

In [ ]:
history

Now let the framework handle it. A **checkpointer** saves the conversation per `thread_id`, so we never touch the list.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

def ask(agent, text, thread="1"):
    result = agent.invoke({"messages": [{"role": "user", "content": text}]},
                          {"configurable": {"thread_id": thread}})
    return result["messages"][-1].content

def chat(agent, thread):
    while True:
        user = input("You: ")
        if user == "quit": break
        print("You:", user)
        assistant_response = ask(agent, user, thread)
        print("AI:", assistant_response)

agent = create_agent(MODEL, checkpointer=InMemorySaver())
chat(agent, thread="1") 

In [ ]:
chat(agent, thread="1")    

## Stage 3: RAG — give it *our* knowledge
It doesn't know university policy. Load the handbook, chunk it, embed it, and give the agent a **search tool**.

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool

docs = [Document(p.read_text(), metadata={"source": p.name}) for p in Path("../docs").glob("*.md")]
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings())

@tool
def search_handbook(query: str) -> str:
    """Search the university handbook for policies: exams, attendance, grading, fees."""
    return "\n\n".join(d.page_content for d in vector_store.similarity_search(query, k=3))

In [ ]:
agent = create_agent(MODEL, tools=[search_handbook], checkpointer=InMemorySaver(),
                     system_prompt="You are a university assistant. Use the handbook for policy questions.")
ask(agent, "What's the minimum attendance to sit the final exam?")

## Stage 4: Tools — from chatbot to agent
It can *answer*, but it can't *do* anything. Tools are plain functions; the **docstring** is what the model reads.

In [ ]:
@tool
def check_timetable(course_code: str) -> str:
    """Get the class schedule for a course code like CS101."""
    return {"CS101": "Mon/Wed 10:00, Room B2", "AI301": "Tue/Thu 14:00, Lab 4"}.get(course_code, "Not found")

@tool
def book_study_room(room: str, time: str) -> str:
    """Book a library study room at a given time."""
    return f"✅ Booked {room} at {time}"

tools = [search_handbook, check_timetable, book_study_room]
agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver())
ask(agent, "When is AI301? Book me study room 3 an hour before it.")

👉 Open LangSmith and show the trace: model → tool call → tool result → model.

## Stage 5: Human-in-the-loop
Actions with consequences should ask first. One line of middleware.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

agent = create_agent(MODEL, tools=tools, checkpointer=InMemorySaver(),
                     middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_study_room": True})])

config = {"configurable": {"thread_id": "hitl"}}
result = agent.invoke({"messages": [{"role": "user", "content": "Book room 3 at 5pm"}]}, config)
result["__interrupt__"]                    # ⏸️ paused, waiting for a human

In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)   # or "reject"
result["messages"][-1].content